# LeetCode 3707 — Equal Score Substrings bằng Python

Link đề: https://leetcode.com/problems/equal-score-substrings/ · Độ khó: **Easy** · Chủ đề: String, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def scoreBalance(self, s: str) -> bool:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm (`scoreBalance`) đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi chữ thường `s`, điểm của chữ cái là thứ tự trong bảng chữ cái (`a = 1`, ..., `z = 26`); hỏi có cách **cắt thành hai đoạn không rỗng** có tổng điểm bằng nhau không.
2. Dùng `ord()` để đổi chữ cái thành điểm, và hiểu **prefix sum** (tổng tiền tố) để biết tổng của một đoạn trong `O(1)`.
3. Biến bài toán thành điều kiện gọn: tồn tại tiền tố có tổng đúng bằng **một nửa tổng toàn chuỗi**.
4. Viết 4 cách: thử mọi điểm cắt với `sum` trên lát cắt, mảng prefix sum, một lượt chạy với tổng chạy, và hai con trỏ; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: dùng `ord(c)` thẳng (không trừ), chỉ kiểm tra tổng chẵn, chỉ thử điểm cắt giữa chuỗi, và lệch một ở điểm cắt cuối.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ cái thường. **Điểm** của một chuỗi là tổng điểm các chữ cái của nó, trong đó `'a' = 1`, `'b' = 2`, ..., `'z' = 26`.

Hãy xác định có tồn tại chỉ số `i` (`0 <= i < n - 1`) sao cho chia `s` thành hai đoạn **không rỗng** `s[0..i]` và `s[i+1..n-1]` mà **điểm hai đoạn bằng nhau** hay không. Trả về `true` nếu có, ngược lại `false`.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"adcb"` | `true` | Cắt sau `i = 1`: `"ad"` có `1 + 4 = 5`, `"cb"` có `3 + 2 = 5` |
| 2 | `"bace"` | `false` | Tổng `2+1+3+5 = 11` lẻ, mọi cách cắt đều lệch |

Constraints: `2 <= s.length <= 100`, `s` chỉ gồm chữ cái thường tiếng Anh.

In [1]:
def score(t):
    return sum(ord(c) - ord('a') + 1 for c in t)

for s in ["adcb", "bace"]:
    print(s, "-> điểm từng chữ:", [ord(c) - 96 for c in s], "| tổng =", score(s))
    for i in range(len(s) - 1):
        print(f"   cắt sau i={i}: {s[:i + 1]!r}={score(s[:i + 1])} | {s[i + 1:]!r}={score(s[i + 1:])}")

adcb -> điểm từng chữ: [1, 4, 3, 2] | tổng = 10
   cắt sau i=0: 'a'=1 | 'dcb'=9
   cắt sau i=1: 'ad'=5 | 'cb'=5
   cắt sau i=2: 'adc'=8 | 'b'=2
bace -> điểm từng chữ: [2, 1, 3, 5] | tổng = 11
   cắt sau i=0: 'b'=2 | 'ace'=9
   cắt sau i=1: 'ba'=3 | 'ce'=8
   cắt sau i=2: 'bac'=6 | 'e'=5


## 3. Kiến thức nền

### 3.1. `ord()` và `chr()`: chữ cái <-> mã số

`ord('a') = 97`, `ord('b') = 98`, ... Vì bảng mã của chữ thường **liên tiếp**, điểm của chữ `c` là `ord(c) - ord('a') + 1` (hay `ord(c) - 96`). `chr(97)` là phép ngược lại.

In [2]:
print(ord('a'), ord('b'), ord('z'))
print(chr(97), chr(122))
print([ord(c) - ord('a') + 1 for c in "az"])

97 98 122
a z
[1, 26]


### 3.2. Lát cắt chuỗi `s[a:b]` và quy tắc nửa mở

`s[a:b]` lấy các ký tự chỉ số `a, a+1, ..., b-1` (**không** gồm `b`). Vậy `s[:i+1]` là `i + 1` ký tự đầu (chỉ số `0..i`) và `s[i+1:]` là phần còn lại. Chú ý cộng thêm `1` để ký tự ở chỉ số `i` thuộc **đoạn trái**.

In [3]:
s = "adcb"
i = 1
print(s[:i + 1], "|", s[i + 1:])
print(s[:0] == "", s[4:] == "")          # lát cắt rỗng khi cắt ở biên

ad | cb
True True


### 3.3. Tổng tiền tố (prefix sum)

Mảng `pre` với `pre[0] = 0` và `pre[k] = a[0] + ... + a[k-1]`. Khi đó tổng đoạn `a[l..r-1]` là `pre[r] - pre[l]` trong `O(1)`. Ở bài này ta chỉ cần tiền tố bắt đầu từ đầu chuỗi nên tổng đoạn trái là `pre[i+1]`, đoạn phải là `pre[n] - pre[i+1]`. Thư viện `itertools.accumulate` tạo prefix sum sẵn.

In [4]:
from itertools import accumulate
a = [1, 4, 3, 2]
pre = [0] + list(accumulate(a))
print(pre)                        # [0, 1, 5, 8, 10]
print("tổng a[1..2] =", pre[3] - pre[1], "=", sum(a[1:3]))

[0, 1, 5, 8, 10]
tổng a[1..2] = 7 = 7


### 3.4. Biến `total`, `left` và điều kiện `left == total - left`

Nếu `left` là tổng đoạn trái và `total` là tổng cả chuỗi thì tổng đoạn phải là `total - left`. Hai đoạn bằng nhau khi `left == total - left`, tức `2 * left == total`. Dùng phép nhân đôi tránh được chia và số lẻ.

In [5]:
total, left = 10, 5
print(left == total - left, 2 * left == total)
total, left = 11, 5
print(left == total - left, 2 * left == total)

True True
False False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hình dung chuỗi như một thanh có các "khối" nặng `1..26` xếp hàng. Ta tìm một điểm đặt dao cắt sao cho hai bên nặng bằng nhau — như tìm **điểm cân bằng**.

Tính tay `"adcb"`: điểm `[1, 4, 3, 2]`, tổng `10`, một nửa là `5`.

```
cắt sau a  : 1 | 4+3+2 = 9
cắt sau ad : 1+4 = 5 | 3+2 = 5   -> bằng!
cắt sau adc: 8 | 2
```

Với `"bace"`: điểm `[2, 1, 3, 5]`, tổng `11` (lẻ) -> không thể chia đôi đều. Trực giác: nếu tổng lẻ thì chắc chắn `false`; nếu tổng chẵn, ta cần một tiền tố có tổng **đúng bằng nửa tổng**.

### Bảng tiền tố của `"adcb"` và `"bace"`

| s | điểm | tiền tố (sau mỗi ký tự) | tổng | nửa tổng | có tiền tố bằng nửa? |
|---|---|---|---|---|---|
| `adcb` | 1, 4, 3, 2 | 1, 5, 8, 10 | 10 | 5 | có (5) |
| `bace` | 2, 1, 3, 5 | 2, 3, 6, 11 | 11 | 5.5 | không |

Chú ý: tiền tố cuối cùng (toàn chuỗi) bằng `total` và không được tính là điểm cắt hợp lệ vì đoạn phải rỗng; nhưng nó cũng không thể bằng `total / 2` vì mọi điểm đều dương.

In [6]:
from itertools import accumulate
for s in ["adcb", "bace", "aab", "zz", "ab"]:
    pts = [ord(c) - 96 for c in s]
    pre = list(accumulate(pts))
    total = pre[-1]
    print(f"{s:5} điểm={pts} tiền tố={pre} tổng={total} -> có tiền tố = tổng/2:", any(2 * p == total for p in pre))

adcb  điểm=[1, 4, 3, 2] tiền tố=[1, 5, 8, 10] tổng=10 -> có tiền tố = tổng/2: True
bace  điểm=[2, 1, 3, 5] tiền tố=[2, 3, 6, 11] tổng=11 -> có tiền tố = tổng/2: False
aab   điểm=[1, 1, 2] tiền tố=[1, 2, 4] tổng=4 -> có tiền tố = tổng/2: True
zz    điểm=[26, 26] tiền tố=[26, 52] tổng=52 -> có tiền tố = tổng/2: True
ab    điểm=[1, 2] tiền tố=[1, 3] tổng=3 -> có tiền tố = tổng/2: False


## 5. Cách 1 — Brute Force / baseline

Thử **mọi** điểm cắt `i` từ `0` đến `n - 2`. Với mỗi `i`, tính điểm hai lát cắt `s[:i+1]` và `s[i+1:]` từ đầu bằng `sum`, rồi so sánh.

Nút thắt: mỗi điểm cắt tính lại tổng từ đầu, tốn `O(n)`; tổng cộng `O(n^2)`. Với `n <= 100` vẫn tức thì, nhưng việc **tính lặp** là điều ta có thể tránh.

In [7]:
class SolutionBrute:
    def scoreBalance(self, s):
        n = len(s)
        for i in range(n - 1):                      # điểm cắt sau chỉ số i, đảm bảo hai đoạn không rỗng
            left = sum(ord(c) - 96 for c in s[:i + 1])
            right = sum(ord(c) - 96 for c in s[i + 1:])
            if left == right:
                return True
        return False

print(SolutionBrute().scoreBalance("adcb"), SolutionBrute().scoreBalance("bace"))

True False


**Độ phức tạp:** Time `O(n^2)` (n điểm cắt, mỗi điểm cộng `O(n)`), Space `O(n)` cho các lát cắt.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** không cần tính lại tổng. Khi dời điểm cắt sang phải một ký tự, đoạn trái **thêm** điểm ký tự đó và đoạn phải **bớt** đúng điểm ấy. Nếu biết `total` từ đầu thì `right = total - left`.

**Quy tắc "nếu… thì…":**

- Nếu `2 * left == total` tại một điểm cắt hợp lệ -> trả `True` ngay.
- Nếu `total` lẻ -> chắc chắn `False` (tiết kiệm công, nhưng không bắt buộc kiểm tra riêng).
- Vì mọi điểm đều dương nên `left` **tăng nghiêm ngặt**; khi `2 * left > total` thì mọi điểm cắt sau đó chỉ lệch thêm -> có thể dừng sớm (`False`).

Điều kiện hợp lệ của điểm cắt (cả hai đoạn không rỗng) là `i` chạy `0..n-2`; điểm cắt sau ký tự cuối bị loại.

In [8]:
# Kiểm chứng: left tăng nghiêm ngặt nên có đúng nhiều nhất một điểm cắt thỏa 2*left == total
import random
random.seed(0)
for _ in range(2000):
    s = "".join(random.choice("abc") for _ in range(random.randint(2, 10)))
    total = sum(ord(c) - 96 for c in s)
    left, hits = 0, 0
    for c in s[:-1]:
        left += ord(c) - 96
        hits += (2 * left == total)
    assert hits <= 1
print("mỗi chuỗi có tối đa một điểm cắt cân bằng")

mỗi chuỗi có tối đa một điểm cắt cân bằng


## 7. Các cách tiếp theo

### Cách 2 — Mảng prefix sum

Dựng `pre` một lần (`O(n)`). Với mỗi điểm cắt `i`, đoạn trái là `pre[i+1]`, đoạn phải là `pre[n] - pre[i+1]`. Cách này tách "chuẩn bị dữ liệu" và "truy vấn", dễ mở rộng khi bài đòi hỏi nhiều truy vấn.

In [9]:
from itertools import accumulate

class SolutionPrefix:
    def scoreBalance(self, s):
        n = len(s)
        pre = [0] + list(accumulate(ord(c) - 96 for c in s))     # pre[k] = điểm k ký tự đầu
        for i in range(n - 1):
            left = pre[i + 1]
            right = pre[n] - pre[i + 1]
            if left == right:
                return True
        return False

print(SolutionPrefix().scoreBalance("adcb"), SolutionPrefix().scoreBalance("bace"))

True False


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho mảng `pre`.

### Cách 3 — Một lượt, tổng chạy `left` (tối ưu bộ nhớ)

Tính `total` trước bằng `sum(...)`, rồi duyệt các ký tự **trừ ký tự cuối** và cộng dồn `left`; kiểm tra `2 * left == total`. Không cần mảng phụ -> Space `O(1)`.

In [10]:
class SolutionRunning:
    def scoreBalance(self, s):
        total = sum(ord(c) - 96 for c in s)
        left = 0
        for c in s[:-1]:                 # bỏ ký tự cuối: đoạn phải không được rỗng
            left += ord(c) - 96
            if 2 * left == total:
                return True
        return False

print(SolutionRunning().scoreBalance("adcb"), SolutionRunning().scoreBalance("bace"))

True False


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (lát `s[:-1]` tạo bản sao nhỏ `O(n)` nếu tính chặt; dùng `range(n - 1)` để đúng `O(1)`).

### Cách 4 — Hai con trỏ từ hai đầu

Đặt `i = 0`, `j = n - 1`, `L` = điểm `s[0]`, `R` = điểm `s[n-1]`. Khi hai đoạn chưa chạm nhau (`j - i > 1`): nếu `L <= R` thì mở rộng đoạn trái (`i += 1; L += điểm(s[i])`), ngược lại mở rộng đoạn phải (`j -= 1; R += điểm(s[j])`). Cuối cùng so `L == R`. Ý tưởng: luôn **bổ sung vào bên nhẹ hơn**; vì điểm dương nên không bao giờ vượt quá điểm cắt thật nếu nó tồn tại.

In [11]:
class SolutionTwoPointers:
    def scoreBalance(self, s):
        n = len(s)
        i, j = 0, n - 1
        L, R = ord(s[0]) - 96, ord(s[-1]) - 96
        while j - i > 1:
            if L <= R:
                i += 1
                L += ord(s[i]) - 96
            else:
                j -= 1
                R += ord(s[j]) - 96
        return L == R

print(SolutionTwoPointers().scoreBalance("adcb"), SolutionTwoPointers().scoreBalance("bace"))

True False


**Độ phức tạp:** Time `O(n)` (mỗi bước dịch một con trỏ, tổng `n - 2` bước), Space `O(1)`.

## 8. TRACE từng bước

Trace `s = "adcb"` bằng **Cách 3**: `total = 10`.

| ký tự | điểm | left sau khi cộng | `2*left` | `2*left == 10`? |
|---|---|---|---|---|
| `a` | 1 | 1 | 2 | không |
| `d` | 4 | 5 | 10 | **có** -> `True` |

Trace Cách 4 với `"bace"`: `i=0, j=3`, `L=2, R=5`. `L <= R` -> `i=1`, `L=3`. `L <= R` -> `i=2`, `L=6`. Lúc này `j - i = 1` dừng; `6 != 5` -> `False`.

In [12]:
def trace_running(s):
    total = sum(ord(c) - 96 for c in s)
    print(f"s={s!r} total={total}")
    left = 0
    for idx in range(len(s) - 1):
        left += ord(s[idx]) - 96
        print(f"  idx={idx} ký tự={s[idx]!r} left={left} right={total - left} cân bằng? {left == total - left}")
        if 2 * left == total:
            print("  => True")
            return True
    print("  => False")
    return False

assert trace_running("adcb") is True
assert trace_running("bace") is False

s='adcb' total=10
  idx=0 ký tự='a' left=1 right=9 cân bằng? False
  idx=1 ký tự='d' left=5 right=5 cân bằng? True
  => True
s='bace' total=11
  idx=0 ký tự='b' left=2 right=9 cân bằng? False
  idx=1 ký tự='a' left=3 right=8 cân bằng? False
  idx=2 ký tự='c' left=6 right=5 cân bằng? False
  => False


In [13]:
def trace_two_pointers(s):
    n = len(s)
    i, j = 0, n - 1
    L, R = ord(s[0]) - 96, ord(s[-1]) - 96
    print(f"s={s!r}: i={i} j={j} L={L} R={R}")
    while j - i > 1:
        if L <= R:
            i += 1
            L += ord(s[i]) - 96
            print(f"  L<=R: mở rộng trái -> i={i} L={L}")
        else:
            j -= 1
            R += ord(s[j]) - 96
            print(f"  L>R : mở rộng phải -> j={j} R={R}")
    print(f"  kết thúc: L={L} R={R} -> {L == R}")
    return L == R

assert trace_two_pointers("adcb") is True
assert trace_two_pointers("bace") is False

s='adcb': i=0 j=3 L=1 R=2
  L<=R: mở rộng trái -> i=1 L=5
  L>R : mở rộng phải -> j=2 R=5
  kết thúc: L=5 R=5 -> True
s='bace': i=0 j=3 L=2 R=5
  L<=R: mở rộng trái -> i=1 L=3
  L<=R: mở rộng trái -> i=2 L=6
  kết thúc: L=6 R=5 -> False


## 9. Code nộp LeetCode

In [14]:
from typing import List, Optional

class Solution:
    def scoreBalance(self, s: str) -> bool:
        total = sum(ord(c) - 96 for c in s)
        left = 0
        for i in range(len(s) - 1):          # điểm cắt sau i, đoạn phải không rỗng
            left += ord(s[i]) - 96
            if 2 * left == total:
                return True
        return False

## 10. Vì sao thuật toán đúng

- **Đúng định nghĩa:** tại điểm cắt sau chỉ số `i`, đoạn trái có điểm `left = pre[i+1]` và đoạn phải có điểm `total - left`. Hai điểm bằng nhau khi và chỉ khi `left = total - left`, tức `2 * left = total`.
- **Không rỗng:** `i` chạy từ `0` đến `n - 2`, nên đoạn trái có ít nhất 1 ký tự và đoạn phải cũng có ít nhất 1 ký tự.
- **Không bỏ sót:** ta thử **mọi** điểm cắt hợp lệ, nên nếu tồn tại một điểm cân bằng thì sẽ gặp.
- **Hai con trỏ:** giả sử có điểm cắt thật tại `p` (trái `s[0..p-1]`, phải `s[p..]`). Invariant: `i <= p - 1` và `j >= p`. Nếu `i < p - 1` thì `L < L_p = R_p`; nếu `j > p` thì `R < R_p`. Khi `L <= R` ta dời `i` (vẫn `<= p-1`); khi `L > R` ta dời `j`, và `j > p` vì nếu `j = p` thì `R = R_p > L` mâu thuẫn. Do đó khi dừng (`j = i + 1`) ta có `i = p - 1`, `j = p` và `L = R`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Thử mọi điểm cắt, `sum` lát cắt | `O(n^2)` | `O(n)` |
| Mảng prefix sum | `O(n)` | `O(n)` |
| Tổng chạy `left` | `O(n)` | `O(1)` |
| Hai con trỏ | `O(n)` | `O(1)` |

`n = len(s) <= 100`. Cả bốn cách đều rất nhanh, nhưng cách 3 là gọn và tiết kiệm nhất. Đây là một dạng "tìm điểm cân bằng" giống bài *Find Pivot Index*.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 2`), phản ví dụ về tổng chẵn mà vẫn `False`, và so sánh **tất cả cách** trên nhiều input.

In [15]:
impls = [SolutionBrute(), SolutionPrefix(), SolutionRunning(), SolutionTwoPointers(), Solution()]

cases = [
    ("adcb", True), ("bace", False),
    ("aa", True), ("ab", False), ("zz", True), ("az", False),
    ("aab", True), ("baa", True), ("abab", True),
    ("ae", False), ("abcd", False), ("aaaa", True), ("zzzzzzzz", True),
]
for s, exp in cases:
    for impl in impls:
        got = impl.scoreBalance(s)
        assert got == exp, (type(impl).__name__, s, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 13 ca


In [16]:
import random
random.seed(3707)

def oracle(s):
    sc = lambda t: sum(ord(c) - 96 for c in t)
    return any(sc(s[:i]) == sc(s[i:]) for i in range(1, len(s)))

def make_true(rng):
    # xây chuỗi chắc chắn True: đoạn trái ngẫu nhiên, đoạn phải có tổng điểm bằng đoạn trái
    left = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(rng.randint(1, 6)))
    target = sum(ord(c) - 96 for c in left)
    right = ""
    while target:
        v = rng.randint(1, min(26, target))
        right += chr(96 + v)
        target -= v
    return left + right

cnt_true = 0
for t in range(800):
    if t % 4 == 0:
        s = make_true(random)
    else:
        alphabet = random.choice(["ab", "abc", "abcdefghijklmnopqrstuvwxyz"])
        s = "".join(random.choice(alphabet) for _ in range(random.randint(2, 12)))
    s = s[:100]
    exp = oracle(s)
    cnt_true += exp
    for impl in impls:
        assert impl.scoreBalance(s) == exp, (type(impl).__name__, s, exp)
print("800 test ngẫu nhiên (seed=3707) OK; số ca True =", cnt_true)
assert cnt_true > 100

800 test ngẫu nhiên (seed=3707) OK; số ca True = 339


In [17]:
# Chuỗi dài tối đa (n = 100) và vài chuỗi đặc biệt
big_true = "a" * 50 + "a" * 50          # 50 | 50
big_false = "a" * 99 + "b"             # 99 vs ... tổng 101 lẻ
assert all(impl.scoreBalance(big_true) for impl in impls)
assert not any(impl.scoreBalance(big_false) for impl in impls)
assert all(impl.scoreBalance("z" * 100) for impl in impls)
assert all(not impl.scoreBalance("a" * 99 + "z") for impl in impls)     # tổng 99+26 = 125 lẻ
print("n = 100 OK")

n = 100 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `ord(c)` mà quên trừ `96`

`ord('a') = 97`, không phải `1`. Khi hai đoạn có độ dài khác nhau, phần dư `96 * độ dài` làm lệch kết quả.

In [18]:
def wrong_ord(s):
    total = sum(ord(c) for c in s)                  # SAI: thiếu - 96
    left = 0
    for i in range(len(s) - 1):
        left += ord(s[i])
        if 2 * left == total:
            return True
    return False

print(wrong_ord("aab"), "!=", Solution().scoreBalance("aab"))     # aa|b = 2 | 2 nhưng ord: 194 | 98
assert wrong_ord("aab") is False and Solution().scoreBalance("aab") is True

False != True


### Lỗi 2: chỉ kiểm tra tổng chẵn

Tổng chẵn chỉ là **điều kiện cần**, chưa đủ: `"ae"` có tổng `6` chẵn nhưng `1 | 5` không bằng.

In [19]:
def wrong_even_only(s):
    return sum(ord(c) - 96 for c in s) % 2 == 0        # SAI: thiếu tìm điểm cắt

print(wrong_even_only("ae"), "!=", Solution().scoreBalance("ae"))
assert wrong_even_only("ae") is True and Solution().scoreBalance("ae") is False

True != False


### Lỗi 3: chỉ thử cắt ở giữa chuỗi

Điểm cắt cân bằng **không** nhất thiết nằm ở giữa theo độ dài: `"aab"` cân bằng tại `aa | b`.

In [20]:
def wrong_middle_only(s):
    mid = len(s) // 2
    sc = lambda t: sum(ord(c) - 96 for c in t)
    return sc(s[:mid]) == sc(s[mid:])                   # SAI: chỉ một điểm cắt

print(wrong_middle_only("aab"), "!=", Solution().scoreBalance("aab"))
assert wrong_middle_only("aab") is False and Solution().scoreBalance("aab") is True

False != True


### Lỗi 4: lệch một, bỏ sót điểm cắt cuối

`range(n - 2)` chỉ thử `i = 0..n-3`, bỏ điểm cắt `i = n - 2` (đoạn phải chỉ còn ký tự cuối).

In [21]:
def wrong_range(s):
    total = sum(ord(c) - 96 for c in s)
    left = 0
    for i in range(len(s) - 2):                         # SAI: phải là len(s) - 1
        left += ord(s[i]) - 96
        if 2 * left == total:
            return True
    return False

print(wrong_range("aab"), "!=", Solution().scoreBalance("aab"))
assert wrong_range("aab") is False and Solution().scoreBalance("aab") is True

False != True


### Lỗi 5: đếm ký tự điểm cắt vào cả hai đoạn

Dùng `s[:i+1]` và `s[i:]` làm ký tự `s[i]` bị tính hai lần.

In [22]:
def wrong_overlap(s):
    sc = lambda t: sum(ord(c) - 96 for c in t)
    return any(sc(s[:i + 1]) == sc(s[i:]) for i in range(1, len(s)))      # SAI: s[i] bị đếm hai lần

print(wrong_overlap("aa"), wrong_overlap("zz"), "vs", Solution().scoreBalance("aa"))
assert wrong_overlap("aba") is True and Solution().scoreBalance("aba") is False

False False vs True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia thành hai phần có tổng bằng nhau" -> **prefix sum** và điều kiện `2 * prefix == total`.
- Mọi điểm đều dương nên prefix tăng nghiêm ngặt -> chỉ có tối đa một điểm cắt, có thể dừng sớm khi `2 * left > total`.
- `ord(c) - 96` cho chữ thường (`ord(c) - 64` cho chữ hoa).
- Nhớ điểm cắt phải chừa **ít nhất một ký tự** cho bên phải: vòng lặp `range(n - 1)`.
- Hai con trỏ cũng hay dùng cho bài "cân bằng" khi các giá trị dương.
- Tự kiểm tra nhanh: ví dụ nhỏ `"aa"` phải `True`, `"ab"` phải `False`.

## 15. Biến thể của bài

1. **Find Pivot Index** (LeetCode 724): tìm chỉ số mà tổng bên trái bằng tổng bên phải (phần tử pivot không tính).
2. **Number of Ways to Split Array** (2270): đếm điểm cắt mà tổng trái `>=` tổng phải.
3. **Find the Middle Index in Array** (1991).
4. **Maximum Score After Splitting a String** (1422): chia nhị phân thành hai phần, điểm = số `0` trái + số `1` phải.
5. **Split Array with Equal Sum** (548): cắt thành bốn đoạn bằng nhau (khó hơn).
6. Đếm **số** điểm cắt cân bằng khi điểm có thể bằng `0` hoặc âm (khi đó có thể có nhiều điểm).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Thử mọi điểm cắt + `sum` | tính lại tổng mỗi lần | `O(n^2)` | `O(n)` | Viết oracle để kiểm tra |
| Mảng prefix sum | `pre[i+1]` và `pre[n] - pre[i+1]` | `O(n)` | `O(n)` | Cần nhiều truy vấn tổng đoạn |
| Tổng chạy `left` | cộng dồn và so với `total` | `O(n)` | `O(1)` | Nộp bài, gọn nhất |
| Hai con trỏ | thêm vào bên nhẹ hơn | `O(n)` | `O(1)` | Khi điểm dương, muốn tránh tính `total` |

Nộp cách 3.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().scoreBalance("abcz")` ra gì? (Tổng bằng bao nhiêu?)

**Bài 2 (tính tay):** với `s = "bbcd"`, hãy liệt kê điểm từng chữ, tổng, và tìm (nếu có) điểm cắt cân bằng.

**Bài 3 (điền TODO):** hoàn thành `split_index(s)` trả về chỉ số `i` của điểm cắt cân bằng, hoặc `-1` nếu không có.

**Bài 4 (tự viết lại):** viết `score_balance_prefix(s)` bằng `itertools.accumulate` và `any`, **không** dùng vòng `for` tường minh.

**Bài 5 (mini challenge):** viết `count_balanced_splits(s)` trả về **số** điểm cắt cân bằng khi điểm của chữ cái là `ord(c) - 96` nhưng cho phép chuỗi gồm cả ký tự `'*'` có điểm `0` (khi đó có thể nhiều hơn một điểm cắt).

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().scoreBalance("abcz"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def split_index_todo(s):
    # TODO: duyệt i, cộng dồn left, so sánh 2 * left với total; trả về i hoặc -1
    return -1

False


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: điểm `1, 2, 3, 26`, tổng `32`; nửa tổng là `16`. Bài 2: `b=2, b=2, c=3, d=4`, tổng `11`.

**Mức 2 (vừa):** Bài 3: giống code nộp nhưng `return i` thay vì `return True`. Bài 4: `any(2 * p == total for p in pre[:-1])` với `pre = list(accumulate(...))`. Bài 5: đừng `return` sớm, hãy đếm.

**Mức 3 (gần đáp án):** Bài 1: không có tiền tố nào bằng `16` (`1, 3, 6, 32`), nên `False`. Bài 2: tổng lẻ nên không có điểm cắt. Bài 4: bỏ phần tử cuối của `pre` vì điểm cắt sau ký tự cuối không hợp lệ.

## 19. Đáp án tham khảo cho bài tập

In [24]:
from itertools import accumulate

# Bài 1
assert Solution().scoreBalance("abcz") is False

# Bài 2: bbcd -> 2,2,3,4 tổng 11 lẻ -> không có
assert sum(ord(c) - 96 for c in "bbcd") == 11 and Solution().scoreBalance("bbcd") is False

# Bài 3
def split_index(s):
    total = sum(ord(c) - 96 for c in s)
    left = 0
    for i in range(len(s) - 1):
        left += ord(s[i]) - 96
        if 2 * left == total:
            return i
    return -1
assert split_index("adcb") == 1 and split_index("bace") == -1 and split_index("aab") == 1

# Bài 4
def score_balance_prefix(s):
    pre = list(accumulate(ord(c) - 96 for c in s))
    return any(2 * p == pre[-1] for p in pre[:-1])
for s in ["adcb", "bace", "aa", "ab", "aab", "zz"]:
    assert score_balance_prefix(s) == Solution().scoreBalance(s)

# Bài 5: ký tự '*' có điểm 0 nên có thể có nhiều điểm cắt cân bằng
def count_balanced_splits(s):
    pts = [0 if c == '*' else ord(c) - 96 for c in s]
    total, left, cnt = sum(pts), 0, 0
    for i in range(len(pts) - 1):
        left += pts[i]
        cnt += (2 * left == total)
    return cnt
assert count_balanced_splits("a*a") == 2      # a|*a (1|1) và a*|a (1|1)
assert count_balanced_splits("adcb") == 1
assert count_balanced_splits("ab") == 0
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "chia hai phần có tổng bằng nhau":**

```python
total = sum(score(c) for c in s)
left = 0
for i in range(len(s) - 1):      # chừa ít nhất 1 phần tử cho bên phải
    left += score(s[i])
    if 2 * left == total:
        return True
return False
```

**Checklist trước khi submit:**

- [ ] Đổi chữ cái thành điểm đúng: `ord(c) - 96` (không phải `ord(c)`)?
- [ ] Vòng lặp `range(n - 1)` để đoạn phải không rỗng, không bỏ sót điểm cắt cuối?
- [ ] So sánh `2 * left == total` (tránh chia, tránh số lẻ)?
- [ ] Thử `n = 2` (`"aa"` -> `True`, `"ab"` -> `False`) và `n = 100`?
- [ ] Chữ ký đúng `scoreBalance(self, s: str) -> bool`?
- [ ] Đã so sánh với bản brute-force trên chuỗi ngẫu nhiên?